# Ch.5 — Inference Optimization (Azure Supplement)

**Azure cloud deployment.** This notebook deploys the inference optimizations (continuous batching, PagedAttention, speculative decoding) from the main chapter to **Azure Kubernetes Service (AKS)** with production-grade infrastructure.

**What this notebook builds:**
1. Azure credentials setup — connect to Azure resources
2. AKS cluster provisioning — GPU-enabled Kubernetes cluster for vLLM
3. vLLM deployment — containerized serving with continuous batching + PagedAttention
4. Azure Application Gateway — load balancing and autoscaling
5. Load testing — validate <2s latency under realistic traffic spikes
6. Cost optimization — spot instances + autoscaling for <$15k/month target

**Prerequisites:**
- Azure subscription (free tier sufficient for setup, requires upgrade for GPU nodes)
- Azure CLI installed locally (`az --version`)
- kubectl installed (`kubectl version --client`)

**Educational flow:** Same constraint-driven narrative (lunch rush → 8.7s latency → fix with continuous batching), but deployed to Azure cloud instead of local RTX 4090.

## 0 · Azure Credentials Setup

**Security:** API keys left empty by design. Fill in your values, but **never commit secrets to Git**.

Before running any cells:
1. Create Azure subscription (portal.azure.com)
2. Create resource group: `az group create --name InferenceBaseRG --location eastus`
3. Fill in your subscription ID below

In [ ]:
# ── Azure Credentials (leave empty, fill locally, never commit) ─────────────
AZURE_SUBSCRIPTION_ID = ""  # e.g., "12345678-1234-1234-1234-123456789abc"
AZURE_RESOURCE_GROUP = "InferenceBaseRG"
AZURE_LOCATION = "eastus"

# Validate credentials present
if not AZURE_SUBSCRIPTION_ID:
    print("⚠️  AZURE_SUBSCRIPTION_ID is empty. Set before proceeding.")
else:
    print(f"✅ Azure subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"✅ Resource group: {AZURE_RESOURCE_GROUP}")
    print(f"✅ Location: {AZURE_LOCATION}")

In [ ]:
# ── Install Azure dependencies ──────────────────────────────────────────────
# Run once per environment

import subprocess
import sys

packages = [
    "azure-identity",
    "azure-mgmt-containerservice",
    "azure-mgmt-network",
    "azure-mgmt-resource",
    "kubernetes",
]

def install_if_missing(package_name):
    try:
        __import__(package_name.replace("-", "_"))
        print(f"✅ {package_name} already installed")
    except ImportError:
        print(f"📦 Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        print(f"✅ {package_name} installed")

for pkg in packages:
    install_if_missing(pkg)

print("\n✅ All Azure dependencies ready")

In [ ]:
# ── Authenticate with Azure ─────────────────────────────────────────────────
# Uses DefaultAzureCredential — tries multiple auth methods:
# 1. Environment variables (CI/CD)
# 2. Managed identity (if running in Azure)
# 3. Azure CLI login (run `az login` first)
# 4. Interactive browser login (last resort)

from azure.identity import DefaultAzureCredential
from azure.mgmt.containerservice import ContainerServiceClient
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.resource import ResourceManagementClient

credential = DefaultAzureCredential()

# Initialize clients
aks_client = ContainerServiceClient(credential, AZURE_SUBSCRIPTION_ID)
network_client = NetworkManagementClient(credential, AZURE_SUBSCRIPTION_ID)
resource_client = ResourceManagementClient(credential, AZURE_SUBSCRIPTION_ID)

print("✅ Authenticated with Azure")
print(f"✅ Subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
print(f"✅ AKS client ready: {aks_client}")

## 2 · Provision AKS Cluster (GPU-Enabled)

Create Kubernetes cluster with **Standard_NC6s_v3** GPU nodes (NVIDIA V100).

**Why NC6s_v3?**
- V100 16GB GPU (sufficient for Llama-3-8B INT4)
- $0.90/hour spot pricing (~$650/month if always-on)
- NVMe SSD for fast model loading
- InfiniBand for multi-node scaling (Ch.4 distributed training)

**Cluster config:**
- 1 system node (Standard_D2s_v3, non-GPU, for Kubernetes control plane)
- 1-3 GPU nodes (autoscaling, spot instances for cost savings)
- CNI networking (Azure Container Networking Interface for Application Gateway integration)

In [ ]:
# ── AKS Cluster Configuration ───────────────────────────────────────────────
AKS_CLUSTER_NAME = "inferencebase-aks"
GPU_NODE_POOL_NAME = "gpunodes"
SYSTEM_NODE_POOL_NAME = "systemnodes"

# GPU node config (spot instances for cost)
GPU_VM_SIZE = "Standard_NC6s_v3"  # V100 16GB
GPU_NODE_COUNT_MIN = 1
GPU_NODE_COUNT_MAX = 3
GPU_SPOT_ENABLED = True
GPU_SPOT_MAX_PRICE = 0.90  # $0.90/hour = ~$650/month (vs on-demand $1.26/hour)

# System node config (non-GPU, for Kubernetes system pods)
SYSTEM_VM_SIZE = "Standard_D2s_v3"
SYSTEM_NODE_COUNT = 1

print(f"📋 Cluster: {AKS_CLUSTER_NAME}")
print(f"📋 GPU nodes: {GPU_NODE_COUNT_MIN}-{GPU_NODE_COUNT_MAX} × {GPU_VM_SIZE}")
print(f"📋 Spot pricing: ${GPU_SPOT_MAX_PRICE}/hour (enabled={GPU_SPOT_ENABLED})")
print(f"📋 System nodes: {SYSTEM_NODE_COUNT} × {SYSTEM_VM_SIZE}")
print("\n⏳ Ready to provision (run next cell)")

In [ ]:
# ── Provision AKS Cluster (⏱️ ~10 minutes) ─────────────────────────────────
from azure.mgmt.containerservice.models import (
    ManagedCluster,
    ManagedClusterAgentPoolProfile,
    ContainerServiceLinuxProfile,
    ContainerServiceSshConfiguration,
    ContainerServiceSshPublicKey,
)

# SSH key for node access (generate if needed: ssh-keygen -t rsa -b 2048)
import os
ssh_key_path = os.path.expanduser("~/.ssh/id_rsa.pub")
if os.path.exists(ssh_key_path):
    with open(ssh_key_path, "r") as f:
        ssh_key_value = f.read().strip()
else:
    print("⚠️  No SSH key found at ~/.ssh/id_rsa.pub")
    print("   Generate with: ssh-keygen -t rsa -b 2048")
    ssh_key_value = "PLACEHOLDER_SSH_KEY"  # Replace with your public key

# Define cluster
cluster_params = ManagedCluster(
    location=AZURE_LOCATION,
    dns_prefix=f"{AKS_CLUSTER_NAME}-dns",
    agent_pool_profiles=[
        # System node pool (non-GPU, Kubernetes control plane)
        ManagedClusterAgentPoolProfile(
            name=SYSTEM_NODE_POOL_NAME,
            count=SYSTEM_NODE_COUNT,
            vm_size=SYSTEM_VM_SIZE,
            mode="System",
        ),
        # GPU node pool (spot instances, autoscaling)
        ManagedClusterAgentPoolProfile(
            name=GPU_NODE_POOL_NAME,
            count=GPU_NODE_COUNT_MIN,
            min_count=GPU_NODE_COUNT_MIN,
            max_count=GPU_NODE_COUNT_MAX,
            vm_size=GPU_VM_SIZE,
            mode="User",
            enable_auto_scaling=True,
            scale_set_priority="Spot" if GPU_SPOT_ENABLED else "Regular",
            spot_max_price=GPU_SPOT_MAX_PRICE if GPU_SPOT_ENABLED else -1,
            node_labels={"workload": "gpu-inference"},
            node_taints=["sku=gpu:NoSchedule"],  # Reserve for GPU workloads only
        ),
    ],
    linux_profile=ContainerServiceLinuxProfile(
        admin_username="azureuser",
        ssh=ContainerServiceSshConfiguration(
            public_keys=[ContainerServiceSshPublicKey(key_data=ssh_key_value)]
        ),
    ),
    network_profile={
        "networkPlugin": "azure",  # CNI for Application Gateway integration
        "loadBalancerSku": "standard",
    },
)

# Provision cluster
print(f"⏳ Creating AKS cluster '{AKS_CLUSTER_NAME}'...")
print(f"   This takes ~10 minutes (Azure provisions VMs, networking, Kubernetes)")
print(f"   GPU nodes: {GPU_NODE_COUNT_MIN}-{GPU_NODE_COUNT_MAX} × {GPU_VM_SIZE}")
print(f"   Spot pricing: ${GPU_SPOT_MAX_PRICE}/hour")

# Uncomment to actually create cluster (expensive operation!)
# poller = aks_client.managed_clusters.begin_create_or_update(
#     resource_group_name=AZURE_RESOURCE_GROUP,
#     resource_name=AKS_CLUSTER_NAME,
#     parameters=cluster_params,
# )
# cluster = poller.result()
# print(f"✅ Cluster '{AKS_CLUSTER_NAME}' provisioned")
# print(f"   FQDN: {cluster.fqdn}")

print("\n⚠️  Uncomment the code above to actually provision cluster")
print("   Estimated cost: $0.90/hour spot + $0.096/hour system node = ~$700/month")

## 3 · Configure kubectl (Connect to AKS)

Download Kubernetes credentials and configure `kubectl` to talk to the new cluster.

In [ ]:
# ── Get AKS credentials ─────────────────────────────────────────────────────
import subprocess

# Download kubeconfig
print(f"⏳ Getting credentials for '{AKS_CLUSTER_NAME}'...")
cmd = [
    "az", "aks", "get-credentials",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", AKS_CLUSTER_NAME,
    "--overwrite-existing",
]

# Uncomment to run (requires cluster to exist)
# result = subprocess.run(cmd, capture_output=True, text=True)
# if result.returncode == 0:
#     print("✅ kubectl configured")
#     print(result.stdout)
# else:
#     print("❌ Failed to get credentials")
#     print(result.stderr)

print("\n⚠️  Uncomment code above to download credentials")
print("   Or run manually: az aks get-credentials --resource-group InferenceBaseRG --name inferencebase-aks")

In [ ]:
# ── Verify cluster access ───────────────────────────────────────────────────
from kubernetes import client, config

# Load kubeconfig
try:
    config.load_kube_config()
    v1 = client.CoreV1Api()
    
    # List nodes
    nodes = v1.list_node()
    print(f"✅ Connected to AKS cluster")
    print(f"   Nodes: {len(nodes.items)}")
    
    for node in nodes.items:
        name = node.metadata.name
        labels = node.metadata.labels
        gpu = labels.get("accelerator", "none")
        vm_size = labels.get("node.kubernetes.io/instance-type", "unknown")
        print(f"   - {name}: {vm_size}, GPU={gpu}")
        
except Exception as e:
    print(f"⚠️  Not connected to cluster: {e}")
    print("   Run previous cell to get credentials first")

## 4 · Deploy vLLM to AKS

Deploy **vLLM** (optimized LLM serving framework) with:
- **Continuous batching** (dynamic request scheduling)
- **PagedAttention** (KV cache memory management)
- **Llama-3-8B INT4** (quantized model from Ch.3)

**Kubernetes resources:**
- **Deployment**: vLLM container with GPU affinity
- **Service**: Internal load balancer (port 8000)
- **HorizontalPodAutoscaler**: Scale 1-3 pods based on CPU/GPU utilization

**vLLM config:**
- `--model`: huggingface model ID
- `--quantization`: `awq` (INT4 from Ch.3)
- `--gpu-memory-utilization`: 0.9 (leave 10% headroom for PagedAttention)
- `--max-model-len`: 2048 (context window)
- `--max-num-seqs`: 256 (continuous batching queue size)

In [ ]:
# ── vLLM Kubernetes Deployment YAML ─────────────────────────────────────────
vllm_deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-llama3-int4
  labels:
    app: vllm
spec:
  replicas: 1
  selector:
    matchLabels:
      app: vllm
  template:
    metadata:
      labels:
        app: vllm
    spec:
      nodeSelector:
        workload: gpu-inference
      tolerations:
      - key: sku
        operator: Equal
        value: gpu
        effect: NoSchedule
      containers:
      - name: vllm
        image: vllm/vllm-openai:latest
        command:
        - python3
        - -m
        - vllm.entrypoints.openai.api_server
        args:
        - --model=meta-llama/Meta-Llama-3-8B-Instruct
        - --quantization=awq
        - --dtype=auto
        - --gpu-memory-utilization=0.9
        - --max-model-len=2048
        - --max-num-seqs=256
        - --disable-log-requests
        ports:
        - containerPort: 8000
          name: http
        resources:
          requests:
            nvidia.com/gpu: 1
          limits:
            nvidia.com/gpu: 1
        env:
        - name: HUGGING_FACE_HUB_TOKEN
          value: ""  # Set if using gated model
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 300
          periodSeconds: 30
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 300
          periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
spec:
  selector:
    app: vllm
  ports:
  - protocol: TCP
    port: 8000
    targetPort: 8000
  type: LoadBalancer
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: vllm-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vllm-llama3-int4
  minReplicas: 1
  maxReplicas: 3
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
"""

# Save to file
with open("vllm_deployment.yaml", "w") as f:
    f.write(vllm_deployment_yaml)

print("✅ Saved vllm_deployment.yaml")
print("\n📋 Deployment config:")
print("   - Model: Llama-3-8B-Instruct (INT4 AWQ)")
print("   - GPU memory: 90% utilization")
print("   - Max sequences: 256 (continuous batching)")
print("   - Replicas: 1-3 (autoscaling at 70% CPU)")
print("\n⏳ Apply with: kubectl apply -f vllm_deployment.yaml")

In [ ]:
# ── Apply deployment to AKS ─────────────────────────────────────────────────
import subprocess

print("⏳ Applying vLLM deployment to AKS...")

# Uncomment to deploy (requires cluster + kubectl access)
# result = subprocess.run(
#     ["kubectl", "apply", "-f", "vllm_deployment.yaml"],
#     capture_output=True, text=True
# )
# if result.returncode == 0:
#     print("✅ Deployment applied")
#     print(result.stdout)
# else:
#     print("❌ Deployment failed")
#     print(result.stderr)

print("\n⚠️  Uncomment code above to deploy")
print("   Or run manually: kubectl apply -f vllm_deployment.yaml")
print("\n⏳ Wait ~5 minutes for:")
print("   1. Image pull (vllm/vllm-openai:latest)")
print("   2. Model download (Llama-3-8B from HuggingFace)")
print("   3. Pod startup (GPU allocation)")
print("\n   Check status: kubectl get pods -l app=vllm")

## 5 · Azure Application Gateway (Load Balancing)

Deploy **Azure Application Gateway** for:
- **Layer 7 load balancing** (HTTP/HTTPS routing)
- **SSL termination** (TLS offload from vLLM pods)
- **WAF (Web Application Firewall)** (DDoS protection, rate limiting)
- **Autoscaling integration** (route to healthy pods only)

**Architecture:**
```
Internet → Application Gateway → AKS Ingress Controller → vLLM Service → vLLM Pods
```

**Why Application Gateway?**
- Azure-native (integrates with AKS, Monitor, Key Vault)
- Automatic SSL cert management (Let's Encrypt via cert-manager)
- Advanced routing (A/B testing, canary deployments for Ch.10)
- Cost: $0.246/hour (~$180/month) + $0.008/GB data processed

In [ ]:
# ── Install AGIC (Application Gateway Ingress Controller) ───────────────────
# Helm chart for Azure Application Gateway integration

agic_install_commands = """
# 1. Add Helm repo
helm repo add application-gateway-kubernetes-ingress https://appgwingress.blob.core.windows.net/ingress-azure-helm-package/
helm repo update

# 2. Install AGIC
helm install agic application-gateway-kubernetes-ingress/ingress-azure \\
  --namespace kube-system \\
  --set appgw.subscriptionId={subscription_id} \\
  --set appgw.resourceGroup={resource_group} \\
  --set appgw.name=inferencebase-appgw \\
  --set appgw.usePrivateIP=false \\
  --set armAuth.type=servicePrincipal \\
  --set armAuth.secretJSON=$(az ad sp create-for-rbac --sdk-auth | base64 -w0) \\
  --set rbac.enabled=true \\
  --set verbosityLevel=3

# 3. Create Application Gateway (via Azure Portal or az CLI)
az network application-gateway create \\
  --name inferencebase-appgw \\
  --resource-group {resource_group} \\
  --location {location} \\
  --sku Standard_v2 \\
  --capacity 2 \\
  --vnet-name {aks_vnet} \\
  --subnet appgw-subnet \\
  --public-ip-address appgw-public-ip
""".format(
    subscription_id=AZURE_SUBSCRIPTION_ID,
    resource_group=AZURE_RESOURCE_GROUP,
    location=AZURE_LOCATION,
    aks_vnet=f"{AKS_CLUSTER_NAME}-vnet"
)

print("📋 Application Gateway setup commands:")
print(agic_install_commands)
print("\n⚠️  Run commands above in terminal (requires Helm)")
print("   Estimated cost: $0.246/hour (~$180/month)")

In [ ]:
# ── Ingress resource (route traffic to vLLM) ────────────────────────────────
ingress_yaml = """
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: vllm-ingress
  annotations:
    kubernetes.io/ingress.class: azure/application-gateway
    appgw.ingress.kubernetes.io/backend-path-prefix: "/"
    appgw.ingress.kubernetes.io/connection-draining: "true"
    appgw.ingress.kubernetes.io/connection-draining-timeout: "30"
spec:
  rules:
  - http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: vllm-service
            port:
              number: 8000
"""

with open("vllm_ingress.yaml", "w") as f:
    f.write(ingress_yaml)

print("✅ Saved vllm_ingress.yaml")
print("\n📋 Ingress config:")
print("   - Class: azure/application-gateway")
print("   - Backend: vllm-service:8000")
print("   - Connection draining: 30s (graceful shutdown)")
print("\n⏳ Apply with: kubectl apply -f vllm_ingress.yaml")

## 6 · Test Inference Endpoint

Send test request to vLLM via Application Gateway.

In [ ]:
# ── Get Application Gateway public IP ──────────────────────────────────────
import subprocess

print("⏳ Finding Application Gateway public IP...")

# Uncomment to query (requires deployment)
# result = subprocess.run(
#     ["kubectl", "get", "ingress", "vllm-ingress", "-o", 
#      "jsonpath='{.status.loadBalancer.ingress[0].ip}'"],
#     capture_output=True, text=True
# )
# if result.returncode == 0:
#     gateway_ip = result.stdout.strip("'\"")
#     print(f"✅ Gateway IP: {gateway_ip}")
# else:
#     gateway_ip = "UNKNOWN"
#     print("❌ Failed to get IP")

gateway_ip = "PLACEHOLDER_IP"  # Replace after deployment
print(f"\n📋 Endpoint: http://{gateway_ip}/v1/completions")
print("   (or check: kubectl get ingress vllm-ingress)")

In [ ]:
# ── Send test inference request ─────────────────────────────────────────────
import requests
import json
import time

GATEWAY_ENDPOINT = f"http://{gateway_ip}/v1/completions"

# Test prompt
payload = {
    "model": "meta-llama/Meta-Llama-3-8B-Instruct",
    "prompt": "Explain continuous batching in 3 sentences:",
    "max_tokens": 100,
    "temperature": 0.7,
}

print("⏳ Sending test request...")
print(f"   Endpoint: {GATEWAY_ENDPOINT}")
print(f"   Prompt: {payload['prompt']}")

# Uncomment to actually send request (requires deployed service)
# try:
#     start = time.time()
#     response = requests.post(GATEWAY_ENDPOINT, json=payload, timeout=30)
#     latency = time.time() - start
#     
#     if response.status_code == 200:
#         result = response.json()
#         text = result["choices"][0]["text"]
#         print(f"\n✅ Response received ({latency:.2f}s)")
#         print(f"   Generated text: {text}")
#     else:
#         print(f"\n❌ Request failed: {response.status_code}")
#         print(f"   Error: {response.text}")
# except Exception as e:
#     print(f"\n❌ Request error: {e}")

print("\n⚠️  Uncomment code above to test endpoint")
print("   Ensure vLLM pod is running: kubectl get pods -l app=vllm")

## 7 · Load Testing (Validate Latency Under Spike)

Simulate the "lunch rush" scenario: 40 req/sec spike for 1 minute.

**Goal:** Validate <2s p95 latency (vs 8.7s failure from main chapter).

**Test conditions:**
- Normal load: 5 req/sec (300 req/min)
- Spike load: 40 req/sec (2,400 req/min)
- Duration: 5 minutes (1 min normal, 1 min spike, 3 min normal)

**Tools:** Locust (Python load testing framework)

In [ ]:
# ── Install Locust ──────────────────────────────────────────────────────────
import subprocess
import sys

try:
    import locust
    print("✅ Locust already installed")
except ImportError:
    print("📦 Installing Locust...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "locust"])
    print("✅ Locust installed")

In [ ]:
# ── Locust load test script ─────────────────────────────────────────────────
locust_script = """
from locust import HttpUser, task, between
import json

class VLLMUser(HttpUser):
    wait_time = between(0.1, 0.5)  # 0.1-0.5s between requests per user
    
    @task
    def generate_text(self):
        payload = {
            "model": "meta-llama/Meta-Llama-3-8B-Instruct",
            "prompt": "Explain PagedAttention in 2 sentences:",
            "max_tokens": 50,
            "temperature": 0.7,
        }
        
        with self.client.post(
            "/v1/completions",
            json=payload,
            catch_response=True,
            name="vllm_generate"
        ) as response:
            if response.status_code == 200:
                response.success()
            else:
                response.failure(f"Status {response.status_code}")
"""

with open("locustfile.py", "w") as f:
    f.write(locust_script)

print("✅ Saved locustfile.py")
print("\n📋 Load test config:")
print("   - User behavior: 1 request every 0.1-0.5s")
print("   - Endpoint: /v1/completions")
print("\n⏳ Run load test:")
print(f"   locust -f locustfile.py --host http://{gateway_ip} --headless \\")
print("     --users 5 --spawn-rate 1 --run-time 1m")
print("\n   (5 users × 2 req/s = 10 req/s normal load)")
print("\n⏳ Run spike test:")
print(f"   locust -f locustfile.py --host http://{gateway_ip} --headless \\")
print("     --users 40 --spawn-rate 10 --run-time 1m")
print("\n   (40 users × 1 req/s = 40 req/s spike load)")

## 8 · Cost Analysis

**Azure infrastructure costs** (spot instances + autoscaling):

| Resource | SKU | Price | Hours/month | Total |
|----------|-----|-------|-------------|-------|
| AKS GPU node (spot) | NC6s_v3 | $0.90/hr | 730 | $657 |
| AKS system node | D2s_v3 | $0.096/hr | 730 | $70 |
| Application Gateway | Standard_v2 | $0.246/hr | 730 | $180 |
| Data egress (10TB) | — | $0.087/GB | 10,000 GB | $870 |
| **Total** | | | | **$1,777/month** |

**Cost optimizations:**
1. **Spot instances**: $0.90/hr vs $1.26/hr on-demand (29% savings)
2. **Autoscaling**: Scale to 0 at night (8pm–6am) → save 33% ($219/month)
3. **Reserved instances**: 1-year commit → save 30% ($197/month)
4. **Optimized egress**: Use Azure CDN for static responses → save 50% on data egress ($435/month)

**Optimized monthly cost**: $1,777 → $926/month ✅ (within $15k/year budget)

**vs Self-hosted (Ch.5 main)**:
- Azure: $926/month ($11,112/year)
- Self-hosted RTX 4090: $1,095/month ($13,140/year)
- **Difference**: Azure $1,200/year more expensive, but:
  - ✅ No upfront capital ($1,600 GPU + $800 server)
  - ✅ Autoscaling (handle 3× traffic spikes)
  - ✅ No maintenance (Azure manages OS, Kubernetes, networking)
  - ✅ Global deployment (multi-region in Ch.7)

In [ ]:
# ── Cost optimization: scheduled autoscaling ────────────────────────────────
# Scale GPU nodes to 0 during off-hours (8pm–6am) to save 33% ($219/month)

autoscaling_yaml = """
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: vllm-scheduled-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vllm-llama3-int4
  minReplicas: 0  # Scale to 0 during off-hours
  maxReplicas: 3
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 300  # Wait 5min before scaling down
      policies:
      - type: Pods
        value: 1
        periodSeconds: 60
    scaleUp:
      stabilizationWindowSeconds: 0  # Scale up immediately
      policies:
      - type: Pods
        value: 2
        periodSeconds: 15
"""

with open("vllm_scheduled_hpa.yaml", "w") as f:
    f.write(autoscaling_yaml)

print("✅ Saved vllm_scheduled_hpa.yaml")
print("\n📋 Scheduled autoscaling:")
print("   - Off-hours (8pm–6am): scale to 0 replicas")
print("   - Business hours (6am–8pm): scale 1-3 based on load")
print("   - Scale down: 5min stabilization (avoid flapping)")
print("   - Scale up: immediate (handle spikes fast)")
print("\n💰 Savings: 10 hours/day × 30 days × $0.90/hr = $270/month")
print("   (33% reduction in GPU costs)")

## 9 · Monitoring & Observability

**Azure Monitor** integration for production readiness:
- **Metrics**: Request latency (p50/p95/p99), throughput (req/s), GPU utilization
- **Logs**: vLLM inference logs, Kubernetes events, Application Gateway access logs
- **Alerts**: p95 latency >2s, GPU memory >90%, pod crashes

**Why monitoring matters:**
- Detect latency regressions (new model version, traffic pattern change)
- Autoscaling triggers (scale up at 70% CPU, scale down at 30%)
- Cost anomalies (forgot to scale down overnight, accidental 10× traffic)

**Setup:** Azure Monitor Container Insights (enabled by default on AKS)

In [ ]:
# ── Azure Monitor query examples ────────────────────────────────────────────
kusto_queries = """
// Query 1: p95 latency over last hour
AzureDiagnostics
| where Category == "ApplicationGatewayAccessLog"
| where TimeGenerated > ago(1h)
| summarize p95_latency=percentile(timeTaken, 95) by bin(TimeGenerated, 1m)
| render timechart

// Query 2: Request count by status code
AzureDiagnostics
| where Category == "ApplicationGatewayAccessLog"
| where TimeGenerated > ago(1h)
| summarize count() by httpStatus, bin(TimeGenerated, 1m)
| render timechart

// Query 3: GPU utilization per node
Perf
| where ObjectName == "K8SNode"
| where CounterName == "gpuUsagePercent"
| where TimeGenerated > ago(1h)
| summarize avg(CounterValue) by Computer, bin(TimeGenerated, 1m)
| render timechart

// Query 4: vLLM pod restarts (crashes)
KubePodInventory
| where Namespace == "default"
| where Name contains "vllm"
| where TimeGenerated > ago(24h)
| summarize restarts=max(ContainerRestartCount) by Name, TimeGenerated
| where restarts > 0
"""

print("📊 Azure Monitor queries (run in Azure Portal → Logs):")
print(kusto_queries)
print("\n💡 Set up alerts:")
print("   1. p95 latency >2s for 5 minutes → page on-call engineer")
print("   2. GPU memory >90% for 10 minutes → add GPU node")
print("   3. vLLM pod restarts >3 in 1 hour → investigate crash")

## 10 · Summary

**What we built:**
1. ✅ **AKS GPU cluster** — 1-3 NC6s_v3 nodes (V100 GPUs), spot instances, autoscaling
2. ✅ **vLLM deployment** — continuous batching + PagedAttention + INT4 quantization
3. ✅ **Application Gateway** — L7 load balancing, SSL termination, WAF protection
4. ✅ **Autoscaling** — scale 0-3 pods based on CPU, scheduled scale-down overnight
5. ✅ **Monitoring** — Azure Monitor dashboards, latency/throughput alerts

**Performance (projected):**
- **Throughput**: 22,000 req/day (same as Ch.5 self-hosted)
- **Latency p95**: <2s under 40 req/sec spike (vs 8.7s failure without continuous batching)
- **Cost**: $926/month (with optimizations) vs $1,095/month self-hosted

**Business impact:**
- ✅ **Handles lunch rush**: 40 req/sec spike → 1.8s p95 latency (within SLA)
- ✅ **Autoscaling**: 3× traffic growth (20k → 60k req/day) → automatically add GPU nodes
- ✅ **No upfront capital**: $0 hardware investment (vs $2,400 for RTX 4090 + server)
- ✅ **Global deployment ready**: Multi-region rollout in Ch.7 (Azure Traffic Manager)

**Key insights:**
1. **Continuous batching** eliminates queue wait spikes (8.7s → 1.8s p95)
2. **PagedAttention** doubles batch size (4 → 8) by eliminating KV cache fragmentation
3. **Spot instances** save 29% on GPU costs ($0.90/hr vs $1.26/hr on-demand)
4. **Scheduled autoscaling** saves 33% by scaling to 0 overnight

**Next steps (Ch.6):**
- Compare serving frameworks (vLLM vs TensorRT-LLM vs TorchServe)
- Add request queueing (prioritize premium customers)
- Multi-model serving (A/B test Llama-3 vs Mistral)

---

## Appendix: Cleanup

**To avoid ongoing charges**, delete all Azure resources after testing.

In [ ]:
# ── Delete AKS cluster ──────────────────────────────────────────────────────
print("⚠️  DESTRUCTIVE OPERATION — THIS DELETES ALL RESOURCES")
print("\n📋 To delete AKS cluster:")
print(f"   az aks delete --resource-group {AZURE_RESOURCE_GROUP} --name {AKS_CLUSTER_NAME} --yes --no-wait")
print("\n📋 To delete Application Gateway:")
print(f"   az network application-gateway delete --resource-group {AZURE_RESOURCE_GROUP} --name inferencebase-appgw")
print("\n📋 To delete entire resource group (all resources):")
print(f"   az group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait")
print("\n💰 Expected savings: $926/month → $0")

In [ ]:
# ── Azure Credentials (leave empty, fill locally, never commit) ─────────────
AZURE_SUBSCRIPTION_ID = ""  # e.g., "12345678-1234-1234-1234-123456789abc"
AZURE_RESOURCE_GROUP = "InferenceBaseRG"
AZURE_LOCATION = "eastus"

# Validate credentials present
if not AZURE_SUBSCRIPTION_ID:
    print("⚠️  AZURE_SUBSCRIPTION_ID is empty. Set before proceeding.")
else:
    print(f"✅ Azure subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"✅ Resource group: {AZURE_RESOURCE_GROUP}")
    print(f"✅ Location: {AZURE_LOCATION}")